# 03 - Features

This notebook turns every row of `bbq_table.csv` (one row per context + question + answer option) into a feature vector of 1158 numbers for the bias detector.

It runs in **Google Colab with a T4 GPU** and reads and writes files in the team Google Drive folder, so it will not run on a normal laptop as-is.

See [docs/features.md](../docs/features.md) for the full explanation.

## 1. Setup

Install `sentence-transformers`, check that Colab gave us a GPU (the models are much faster on GPU), mount Google Drive, and load `bbq_table.csv`. It should have 69240 rows.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd

table = pd.read_csv("/content/drive/MyDrive/CSE 575 - Fairness/bbq_table.csv")
print(table.shape)
table.head()

## 2. Embeddings

Load the `all-MiniLM-L6-v2` sentence model and turn the context, question and answer of every row into 384 numbers each that describe the meaning of the text.

Many rows share the same text (for example the same context appears with 3 answers), so we embed each unique text only once and then copy the vectors back to every row. This is much faster. The vectors are normalized to length 1, which makes the next step easy.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

In [ ]:
def embed_column(col):
    unique_texts = table[col].unique()
    print(col, "unique texts:", len(unique_texts))
    vecs = emb_model.encode(list(unique_texts), batch_size=256,
                            show_progress_bar=True, normalize_embeddings=True)
    position = {text: i for i, text in enumerate(unique_texts)}
    return vecs[table[col].map(position).values]

ctx_emb = embed_column("context")
q_emb = embed_column("question")
a_emb = embed_column("answer_text")
print(ctx_emb.shape, q_emb.shape, a_emb.shape)

## 3. Similarities

Cosine similarity between context-question, context-answer and question-answer. Because the embeddings are already length 1, the cosine similarity is just the dot product. This gives the detector 3 numbers that say directly how related the texts are.

In [ ]:
sim_cq = (ctx_emb * q_emb).sum(axis=1)
sim_ca = (ctx_emb * a_emb).sum(axis=1)
sim_qa = (q_emb * a_emb).sum(axis=1)
print(sim_cq[:3], sim_ca[:3], sim_qa[:3])

## 4. Statements

The NLI model in the next step needs a full sentence to check against the context, not a question and an answer separately. So we turn each question + answer into one statement, like "Who was the secretary?" + "The woman" → "The woman was the secretary."

First we check how the questions start, because each question type needs its own rule. Almost all start with "Who", and a few start with "Whose", "What group of people" or "Which parents".

In [ ]:
print(table["question"].str.split().str[0].value_counts())

In [ ]:
other = table[~table["question"].str.startswith("Who ")]
print(other.groupby("category")["question"].unique())

In [ ]:
for cat, qs in other.groupby("category")["question"].unique().items():
    print(cat)
    for q in qs:
        print("   ", q)

`make_statement` builds the statement. If the answer is the "unknown" option (`is_unknown`), it makes "It is not known who ..." no matter what the question type is, because the raw unknown answer text gave nonsense like "Not enough information was the secretary." Then there is one rule per question type.

In [ ]:
def make_statement(question, answer, is_unknown):
    q = question.strip().rstrip("?")
    if is_unknown:
        return "It is not known " + q[0].lower() + q[1:] + "."
    if q.startswith("Who "):
        return answer + q[3:] + "."
    if q.startswith("Whose "):
        return answer + "'s" + q[5:] + "."
    if q.startswith("What group of people "):
        return "The group of people that" + q[len("What group of people"):] + " is " + answer + "."
    if q.startswith("Which parents "):
        return answer + q[len("Which parents"):] + "."
    return f"{q}? {answer}."

table["statement"] = [
    make_statement(q, a, u)
    for q, a, u in zip(table["question"], table["answer_text"], table["is_unknown"])
]

Check: print the 3 statements for a few questions of each type to make sure they read like normal sentences.

In [ ]:
# check: first question of each type, all 3 statements
for qid in table.drop_duplicates("question")["question_id"].head(3).tolist() + \
           table[~table["question"].str.startswith("Who ")].drop_duplicates("question")["question_id"].tolist():
    print(table.loc[table["question_id"] == qid, "statement"].tolist())

## 5. NLI

Load the `cross-encoder/nli-deberta-v3-base` NLI model and run it on every (context, statement) pair. The context is the premise and the statement is the hypothesis. The model says whether the context contradicts, supports (entails) or says nothing about (neutral) the statement.

The model outputs raw scores (logits), so we apply softmax to turn them into 3 probabilities that add up to 1. This is the slow step, so we save `nli_probs.npy` to Drive right away.

In [ ]:
from sentence_transformers import CrossEncoder

nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-base", device="cuda")

pairs = list(zip(table["context"], table["statement"]))
logits = nli_model.predict(pairs, batch_size=128, show_progress_bar=True)

probs = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
print(probs.shape)

In [ ]:
np.save("/content/drive/MyDrive/CSE 575 - Fairness/nli_probs.npy", probs)

## 6. NLI checks

Some quick checks that the NLI output makes sense:
- print `id2label` to see which column is which (for this model: 0 = contradiction, 1 = entailment, 2 = neutral)
- look at the secretary example (the first 6 rows, ambiguous and disambiguated)
- average NLI probabilities grouped by `context_condition`, `is_unknown` and `is_correct`, to see if NLI separates right from wrong answers

In [ ]:
print(nli_model.model.config.id2label)

check = table[["context_condition", "statement"]].head(6).copy()
check[["p0", "p1", "p2"]] = probs[:6].round(2)
check

In [ ]:
nli = pd.DataFrame(probs, columns=["nli_contra", "nli_entail", "nli_neutral"])
check = pd.concat([table[["context_condition", "is_unknown", "is_correct"]], nli], axis=1)
check.groupby(["context_condition", "is_unknown", "is_correct"])[["nli_contra", "nli_entail", "nli_neutral"]].mean().round(2)

## 7. Combine and save

Stack everything into one matrix `X` of shape 69240 x 1158 (float32): context embedding, question embedding, answer embedding, the 3 similarities and the 3 NLI probabilities.

Save `features.npy` and `features_meta.csv` to Drive. Row i of the CSV goes with row i of `features.npy`. They go to Drive, not the repo, because `features.npy` is about 320 MB.

In [ ]:
X = np.hstack([
    ctx_emb, q_emb, a_emb,
    sim_cq[:, None], sim_ca[:, None], sim_qa[:, None],
    probs,
]).astype(np.float32)

print(X.shape)

In [ ]:
folder = "/content/drive/MyDrive/CSE 575 - Fairness/"

np.save(folder + "features.npy", X)

meta = table[["row_id", "split", "is_biased", "category", "context_condition", "statement"]]
meta.to_csv(folder + "features_meta.csv", index=False)

print("saved")